In [1]:
import requests
import pandas as pd
import sqlite3

# =============================================================================
# 1. DESCOBRINDO O LIMITE DINAMICAMENTE (Fugindo dos Magic Numbers)
# =============================================================================
# O endpoint /rodadas traz o planejamento completo do campeonato
url_rodadas = "https://api.cartola.globo.com/rodadas"
resposta_rodadas = requests.get(url_rodadas)

if resposta_rodadas.status_code == 200:
    # A resposta é uma lista. O tamanho dela é o total exato de rodadas do campeonato!
    total_rodadas = len(resposta_rodadas.json())
    print(f"🔍 Campeonato identificado com um limite dinâmico de {total_rodadas} rodadas.")
else:
    print("❌ Erro ao acessar a API de rodadas. Utilizando limite padrão de 38.")
    total_rodadas = 38  # Fallback seguro

# =============================================================================
# 2. EXTRAÇÃO EM LOOP (Carga Histórica / Full Load)
# =============================================================================
url_base = "https://api.cartola.globo.com/partidas"
dados_fatos = []

print("🚀 Iniciando a Carga Completa para rastrear jogos pendentes e atualizações...")

# Varrendo de 1 até o total exato de rodadas que descobrimos acima
for rodada in range(1, total_rodadas + 1):
    url_rodada = f"{url_base}/{rodada}"
    resposta_rodada = requests.get(url_rodada)
    
    if resposta_rodada.status_code == 200:
        jogos = resposta_rodada.json().get('partidas', [])
        
        for jogo in jogos:
            linha = {
                "id_partida": jogo.get('partida_id'),
                "rodada": rodada,
                "data_jogo": jogo.get('partida_data'),
                "id_time_mandante": jogo.get('clube_casa_id'),
                "id_time_visitante": jogo.get('clube_visitante_id'),
                "gols_mandante": jogo.get('placar_oficial_mandante'),
                "gols_visitante": jogo.get('placar_oficial_visitante'),
                "valida_pro_cartola": jogo.get('valida') # Dica útil para tratar W.O ou adiamentos futuramente
            }
            dados_fatos.append(linha)
    
    print(f"⏳ Rodada {rodada}/{total_rodadas} processada...")

# =============================================================================
# 3. TRANSFORMAÇÃO E CARGA
# =============================================================================
if dados_fatos:
    df_fatos = pd.DataFrame(dados_fatos)
    
    try:
        with sqlite3.connect('banco_brasileirao.db') as conexao:
            # O 'replace' garante a idempotência: apaga a antiga e insere a nova atualizada
            df_fatos.to_sql(name='fato_partidas', con=conexao, if_exists='replace', index=False)
        print(f"✅ Sucesso! {len(df_fatos)} partidas empilhadas e salvas na fato_partidas.")
    except Exception as erro:
        print(f"❌ Erro ao salvar no banco de dados: {erro}")

🔍 Campeonato identificado com um limite dinâmico de 38 rodadas.
🚀 Iniciando a Carga Completa para rastrear jogos pendentes e atualizações...
⏳ Rodada 1/38 processada...
⏳ Rodada 2/38 processada...
⏳ Rodada 3/38 processada...
⏳ Rodada 4/38 processada...
⏳ Rodada 5/38 processada...
⏳ Rodada 6/38 processada...
⏳ Rodada 7/38 processada...
⏳ Rodada 8/38 processada...
⏳ Rodada 9/38 processada...
⏳ Rodada 10/38 processada...
⏳ Rodada 11/38 processada...
⏳ Rodada 12/38 processada...
⏳ Rodada 13/38 processada...
⏳ Rodada 14/38 processada...
⏳ Rodada 15/38 processada...
⏳ Rodada 16/38 processada...
⏳ Rodada 17/38 processada...
⏳ Rodada 18/38 processada...
⏳ Rodada 19/38 processada...
⏳ Rodada 20/38 processada...
⏳ Rodada 21/38 processada...
⏳ Rodada 22/38 processada...
⏳ Rodada 23/38 processada...
⏳ Rodada 24/38 processada...
⏳ Rodada 25/38 processada...
⏳ Rodada 26/38 processada...
⏳ Rodada 27/38 processada...
⏳ Rodada 28/38 processada...
⏳ Rodada 29/38 processada...
⏳ Rodada 30/38 processada..